# AMLIOS-X EDA: Raw Dataset and Graph Topology

DevB Phase 8 notebook for validating the hackathon dataset, understanding suspicious-label imbalance, and preparing dashboard/modeling assumptions.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'STUDENT_DATASET'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

sns.set_theme(style='whitegrid', palette='deep')

## Load Dataset Samples

In [ ]:
transactions = pd.read_csv(DATA_DIR / 'transactions.csv')
accounts = pd.read_csv(DATA_DIR / 'accounts.csv')
ml_features = pd.read_csv(DATA_DIR / 'ml_features.csv')
edges = pd.read_csv(DATA_DIR / 'graph_edges.csv')

for name, frame in {
    'transactions': transactions,
    'accounts': accounts,
    'ml_features': ml_features,
    'graph_edges': edges,
}.items():
    print(f'{name}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns')

## Schema, Missingness, and Label Imbalance

In [ ]:
schema = pd.DataFrame({
    'column': transactions.columns,
    'dtype': transactions.dtypes.astype(str).values,
    'missing': transactions.isna().sum().values,
    'missing_pct': (transactions.isna().mean().values * 100).round(2),
})
schema.sort_values('missing_pct', ascending=False).head(20)

In [ ]:
label_counts = ml_features['is_suspicious_tx'].value_counts().rename_axis('label').reset_index(name='count')
label_counts['pct'] = (label_counts['count'] / label_counts['count'].sum() * 100).round(3)
label_counts

## Amount Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(transactions['amount_local_npr'], bins=80, ax=axes[0])
axes[0].set_title('Transaction amount distribution')
axes[0].set_xlabel('Amount local NPR')

sns.histplot(transactions['log_amount'], bins=80, ax=axes[1], color='tab:green')
axes[1].set_title('Log amount distribution')
axes[1].set_xlabel('Log amount')
plt.tight_layout()

## Temporal Patterns

In [ ]:
transactions['date_transaction'] = pd.to_datetime(transactions['date_transaction'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=transactions, x='hour_of_day', ax=axes[0], color='tab:blue')
sns.countplot(data=transactions, x='day_of_week', ax=axes[1], color='tab:orange')
monthly = transactions.set_index('date_transaction').resample('ME').size()
monthly.plot(ax=axes[2], color='tab:green')
axes[0].set_title('Transactions by hour')
axes[1].set_title('Transactions by day of week')
axes[2].set_title('Transactions by month')
plt.tight_layout()

## Cross-Border and Domestic Breakdown

In [ ]:
cross_border = transactions['cross_border_flag'].value_counts().rename(index={0: 'Domestic', 1: 'Cross-border'})
cross_border.plot(kind='bar', figsize=(6, 4), title='Domestic vs cross-border transaction count')
plt.ylabel('Transactions')

## Account Degree and Top Volume Accounts

In [ ]:
out_degree = transactions.groupby('Sender_account').size().rename('out_degree')
in_degree = transactions.groupby('Receiver_account').size().rename('in_degree')
degree = pd.concat([out_degree, in_degree], axis=1).fillna(0)
degree['total_degree'] = degree['out_degree'] + degree['in_degree']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(degree['total_degree'], bins=80, ax=axes[0])
axes[0].set_title('Account degree distribution')

top_volume = transactions.groupby('Sender_account')['amount_local_npr'].sum().nlargest(20)
top_volume.sort_values().plot(kind='barh', ax=axes[1], color='tab:red')
axes[1].set_title('Top 20 sender accounts by volume')
plt.tight_layout()

## Community Size Distribution

In [ ]:
node_features_path = PROCESSED_DIR / 'node_features.csv'
if node_features_path.exists():
    node_features = pd.read_csv(node_features_path)
    community_col = next((col for col in node_features.columns if 'community' in col.lower()), None)
    if community_col:
        node_features[community_col].value_counts().head(50).plot(kind='bar', figsize=(14, 4), title='Largest communities')
    else:
        print('No community column found in node_features.csv yet.')
else:
    print('node_features.csv not available yet; run DevA graph phases before this section.')

## Feature Correlation Heatmap

In [ ]:
numeric_features = ml_features.select_dtypes(include='number')
corr_cols = numeric_features.corr(numeric_only=True)['is_suspicious_tx'].abs().sort_values(ascending=False).head(18).index
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_features[corr_cols].corr(), cmap='vlag', center=0, linewidths=0.4)
plt.title('Top feature correlations around suspicious transaction label')
plt.tight_layout()